Problem Statement:

Create a workflow where a user provides their weight and height. The program should:

- calculate the Body Mass Index (BMI)
- classify the user into a weight category such as Underweight, Normal, or Overweight
- Assume you are his crush. Generate a super negative message based on the weight category

ThinkPot:
This can be solved using LangChain because it follows a linear flow, but we will build the solution using LangGraph instead.
The steps involved in creating this workflow using LangGraph are:

1. Define the state
2. Define the nodes
3. Define the graph
4. Add the nodes to the graph
5. Add edges between the nodes
6. Compile the graph
7. Execute the graph


In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# define state
class BMIState(TypedDict):

    weight_kg: float
    height_m: float
    bmi: float
    category: str
    message: str

In [19]:
# define nodes
def calculate_bmi(state: BMIState) -> BMIState:

    weight = state['weight_kg']
    height = state['height_m']
    bmi = weight/(height**2)
    state['bmi'] = round(bmi, 2)

    return state

def label_bmi(state: BMIState) -> BMIState:

    bmi = state['bmi']

    if bmi < 18.5:
        state["category"] = "Underweight"
    elif 18.5 <= bmi < 25:
        state["category"] = "Normal"
    elif 25 <= bmi < 30:
        state["category"] = "Overweight"
    else:
        state["category"] = "Obese"

    return state

def generate_message(state: BMIState) -> BMIState:
    category = state['category']
    model = ChatOpenAI()
    prompt = f"Assume you are his crush. Generate a super negative message in 3 lines for someone who is categorized as {category} to maintain their health."
    response = model.invoke(prompt)
    state["message"] = response.content

    return state


In [20]:
# define your graph
graph = StateGraph(BMIState)

# add nodes to your graph
graph.add_node('calculate_bmi', calculate_bmi)
graph.add_node('label_bmi', label_bmi)
graph.add_node('generate_message', generate_message)

# add edges to your graph
graph.add_edge(START, 'calculate_bmi')
graph.add_edge('calculate_bmi', 'label_bmi')
graph.add_edge('label_bmi', 'generate_message')
graph.add_edge('generate_message', END)


# compile the graph
workflow = graph.compile()

# execute the graph
initial_state = {'weight_kg':95, 'height_m':1.72}
final_state = workflow.invoke(initial_state)
print(final_state)


{'weight_kg': 95, 'height_m': 1.72, 'bmi': 32.11, 'category': 'Obese', 'message': "I can't be with someone who doesn't take care of themselves. Your weight is a major turn off and shows a lack of self-discipline. Get it together or I'm out."}


## Prompt Chaining

**Prompt chaining** is a technique where an LLM task is split into a sequence of smaller steps. The output of one step is passed as input to the next, so each prompt builds on the previous result. This makes multi-step workflows easier to organize and helps keep each step focused.

### Problem Statement

Create a LangGraph workflow that turns a blog title into a complete blog post. The workflow should:

1. Accept a blog title as input.
2. Ask an LLM to create a detailed outline for that title.
3. Pass the title and generated outline to a second LLM prompt to write the blog post.

### Example

**Input title:** `Rise of AI in India`

**Expected result:** A final state containing the original title, a detailed outline, and a blog post written using that outline.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv

load_dotenv()
model = ChatOpenAI()

In [23]:
class BlogState(TypedDict):

    title: str
    outline: str
    content: str

In [29]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']
    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content
    # update state
    state['outline'] = outline

    return state

def create_blog(state: BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']
    prompt = f'Write a detailed blog in 50 words on the title - {title} using the follwing outline \n {outline}'
    content = model.invoke(prompt).content
    state['content'] = content

    return state

In [30]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', END)

workflow = graph.compile()

intial_state = {'title': 'Rise of AI in India'}
final_state = workflow.invoke(intial_state)
print(final_state['outline'])

I. Introduction
    A. Definition of AI and its impact on various industries 
    B. Overview of the rise of AI in India 
II. Historical background of AI development in India 
    A. Early developments in AI research in India 
    B. Growth of AI startups in India 
III. Current state of AI in India
    A. Major industries adopting AI technology 
    B. Government initiatives to promote AI in India 
    C. Challenges faced by AI companies in India 
IV. Impact of AI on Indian society
    A. Job market disruptions caused by AI 
    B. AI in healthcare and education sectors 
    C. Ethical considerations surrounding AI technology in India 
V. Future prospects of AI in India
    A. Potential growth opportunities for AI companies in India 
    B. Predictions for the future impact of AI on Indian economy 
    C. Role of India in the global AI landscape 
VI. Conclusion
    A. Recap of the rise of AI in India 
    B. Final thoughts on the future of AI technology in India.
